# SeaDronesSee 30 m frozen-encoder training

This notebook builds the **30 m baseline** for Question 3a using the SeaDronesSee Object Detection v2 release.

It deliberately keeps the vision encoder frozen. The notebook:

1. loads the official COCO annotations and continuous altitude metadata;
2. selects observations from **25–35 m above takeoff**;
3. removes ignored annotations and temporally thins near-duplicate video frames;
4. splits by complete source video so neighbouring frames cannot leak between sets;
5. extracts fixed-canvas object features with a frozen DINOv2 encoder;
6. trains and evaluates a linear classifier;
7. saves the split, features, model and metrics needed for the later synthetic-80 m and Procrustes experiments.

This is an **object-classification baseline using ground-truth boxes**. It does not test object localisation and it does not yet make any claim about 80 m performance.

The required altitude subsets can be downloaded from the official release with:

```powershell
python scripts/download_seadronessee_odv2.py --repo . --bands 25:35,70:100
```

Use `--all` instead of `--bands ...` to fetch all labelled train/validation images (approximately 7.4 GB).

In [1]:
from pathlib import Path
import json
import math
import random
import warnings


def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the uav-altitude-feature-alignment repository.")


REPO_ROOT = find_repo_root()
DATASET_ROOT = REPO_ROOT / "data" / "raw" / "seadronessee" / "odv2"
ANNOTATION_DIR = DATASET_ROOT / "annotations"
IMAGE_DIR = DATASET_ROOT / "images"
SPLIT_DIR = REPO_ROOT / "data" / "splits" / "seadronessee"
FEATURE_DIR = REPO_ROOT / "data" / "features" / "seadronessee" / "dinov2_small_30m"
RESULT_DIR = REPO_ROOT / "results" / "seadronessee_30m"

for directory in [SPLIT_DIR, FEATURE_DIR, RESULT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Experimental definition
ALTITUDE_MIN_M = 25.0
ALTITUDE_MAX_M = 35.0
MIN_FRAME_GAP = 30       # approximately one second for 30 fps source video
MIN_BOX_SIDE_PX = 6
CANVAS_SIDE_PX = 640     # fixed source-pixel canvas; object scale is not normalised
CANVAS_FILL = (114, 114, 114)
RANDOM_SEED = 42

# Frozen encoder
MODEL_NAME = "facebook/dinov2-small"
BATCH_SIZE = 32
NUM_WORKERS = 0          # reliable default for Windows/Jupyter

random.seed(RANDOM_SEED)
print(f"Repository: {REPO_ROOT}")
print(f"Dataset:    {DATASET_ROOT}")


Repository: C:\Users\ADMIN\Desktop\fastr\uav-altitude-feature-alignment
Dataset:    C:\Users\ADMIN\Desktop\fastr\uav-altitude-feature-alignment\data\raw\seadronessee\odv2


## Environment

Install the packages in `requirements-research.txt` and select that environment as this notebook's kernel. PyTorch installation may need to be matched to the computer's CUDA version. The following cell fails early with a clear message if a dependency is missing.

In [2]:
try:
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import torch
    import torch.nn.functional as F
    from PIL import Image
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
    )
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from torch.utils.data import DataLoader, Dataset
    from tqdm.auto import tqdm
    from transformers import AutoImageProcessor, AutoModel
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "A research dependency is missing. Activate the project environment and run "
        "`python -m pip install -r requirements-research.txt`, then restart the kernel."
    ) from exc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuntimeError: A research dependency is missing. Activate the project environment and run `python -m pip install -r requirements-research.txt`, then restart the kernel.

## 1. Load the official annotations and select the 30 m band

SeaDronesSee v2 stores altitude under `height_above_takeoff(meter)`. The notebook combines the official train and validation annotations and creates a new split by source video. The official split is retained as metadata but is not used for model separation because frames from the same video can occur in both files.

In [ ]:
def load_coco_objects(annotation_path, source_split):
    with annotation_path.open("r", encoding="utf-8") as handle:
        coco = json.load(handle)

    categories = {
        int(category["id"]): category["name"]
        for category in coco["categories"]
    }
    images = {int(image["id"]): image for image in coco["images"]}
    rows = []

    for annotation in coco["annotations"]:
        category_id = int(annotation["category_id"])
        if category_id == 0:  # dataset's ignored region category
            continue

        image = images[int(annotation["image_id"])]
        meta = image.get("meta") or {}
        source = image.get("source") or {}
        x, y, width, height = map(float, annotation["bbox"])
        altitude = meta.get("height_above_takeoff(meter)")
        gimbal_pitch = meta.get("gimbal_pitch(degrees)")

        rows.append({
            "object_key": f"{source_split}:{annotation['id']}",
            "image_key": f"{source_split}:{image['id']}",
            "source_split": source_split,
            "annotation_id": int(annotation["id"]),
            "image_id": int(image["id"]),
            "file_name": image["file_name"],
            "image_width": int(image["width"]),
            "image_height": int(image["height"]),
            "category_id": category_id,
            "category_name": categories[category_id],
            "bbox_x": x,
            "bbox_y": y,
            "bbox_width": width,
            "bbox_height": height,
            "bbox_area": width * height,
            "altitude_m": float(altitude) if altitude is not None else np.nan,
            "gimbal_pitch_deg": float(gimbal_pitch) if gimbal_pitch is not None else np.nan,
            "drone": source.get("drone", "unknown"),
            "video": source.get("video", f"video_{image.get('video_id', 'unknown')}"),
            "frame_no": int(source.get("frame_no", image.get("frame_index", image["id"]))),
            "image_path": str(IMAGE_DIR / source_split / image["file_name"]),
        })

    return pd.DataFrame(rows), categories


frames = []
category_maps = []
for source_split in ["train", "val"]:
    annotation_path = ANNOTATION_DIR / f"instances_{source_split}.json"
    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Missing {annotation_path}. Run the official downloader command shown at the top of the notebook."
        )
    frame, category_map = load_coco_objects(annotation_path, source_split)
    frames.append(frame)
    category_maps.append(category_map)

objects_all = pd.concat(frames, ignore_index=True)
assert category_maps[0] == category_maps[1], "Train and validation category definitions differ."

objects_30m = objects_all[
    objects_all["altitude_m"].between(ALTITUDE_MIN_M, ALTITUDE_MAX_M, inclusive="both")
    & (objects_all["bbox_width"] >= MIN_BOX_SIDE_PX)
    & (objects_all["bbox_height"] >= MIN_BOX_SIDE_PX)
].copy()
objects_30m["image_exists"] = objects_30m["image_path"].map(lambda value: Path(value).exists())

print(f"All labelled objects: {len(objects_all):,}")
print(f"Objects at {ALTITUDE_MIN_M:.0f}–{ALTITUDE_MAX_M:.0f} m: {len(objects_30m):,}")
print(f"Unique 30 m images: {objects_30m['image_key'].nunique():,}")
print(f"Unique 30 m videos: {objects_30m['video'].nunique():,}")
print(f"Missing selected images: {(~objects_30m['image_exists']).sum():,} object rows")

display(objects_30m.head())


In [ ]:
missing_paths = (
    objects_30m.loc[~objects_30m["image_exists"], "image_path"]
    .drop_duplicates()
    .tolist()
)
if missing_paths:
    print("Example missing files:")
    for value in missing_paths[:10]:
        print(" ", value)
    raise FileNotFoundError(
        "The 30 m annotations are present but some selected images are missing. From the repository root run:\n"
        "python scripts/download_seadronessee_odv2.py --repo . --bands 25:35,70:100"
    )

summary = (
    objects_30m.groupby(["category_name"])
    .agg(objects=("object_key", "count"), images=("image_key", "nunique"), videos=("video", "nunique"))
    .sort_values("objects", ascending=False)
)
display(summary)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(objects_30m.drop_duplicates("image_key"), x="altitude_m", bins=20, ax=axes[0])
axes[0].set_title("Selected image altitudes")
sns.countplot(data=objects_30m, y="category_name", order=summary.index, ax=axes[1])
axes[1].set_title("Objects by category")
sns.histplot(objects_30m.drop_duplicates("image_key"), x="gimbal_pitch_deg", bins=20, ax=axes[2])
axes[2].set_title("Gimbal-pitch metadata")
plt.tight_layout()


## 2. Thin neighbouring video frames

Adjacent frames are highly correlated. All objects in a retained image are kept, but retained images from the same video must be separated by at least `MIN_FRAME_GAP` source frames. This reduces pseudo-replication before the video-level split.

In [ ]:
def select_temporally_spaced_images(frame, minimum_gap):
    image_table = (
        frame[["image_key", "video", "frame_no"]]
        .drop_duplicates("image_key")
        .sort_values(["video", "frame_no", "image_key"])
    )
    selected = []
    for video, group in image_table.groupby("video", sort=True):
        last_frame = -math.inf
        for row in group.itertuples(index=False):
            if row.frame_no - last_frame >= minimum_gap:
                selected.append(row.image_key)
                last_frame = row.frame_no
    return set(selected)


selected_image_keys = select_temporally_spaced_images(objects_30m, MIN_FRAME_GAP)
objects = objects_30m[objects_30m["image_key"].isin(selected_image_keys)].copy()
objects.reset_index(drop=True, inplace=True)

# A class must occur in at least three videos to support disjoint train/validation/test evaluation.
class_video_counts = objects.groupby("category_name")["video"].nunique().sort_values(ascending=False)
eligible_classes = class_video_counts[class_video_counts >= 3].index.tolist()
excluded_classes = class_video_counts[class_video_counts < 3].index.tolist()
if excluded_classes:
    warnings.warn(
        "Excluding classes that occur in fewer than three source videos: "
        + ", ".join(excluded_classes)
    )
objects = objects[objects["category_name"].isin(eligible_classes)].copy()
objects.reset_index(drop=True, inplace=True)

print(f"Images before thinning: {objects_30m['image_key'].nunique():,}")
print(f"Images after thinning/filtering: {objects['image_key'].nunique():,}")
print(f"Objects after thinning/filtering: {len(objects):,}")
print("Videos per retained class:")
display(class_video_counts.loc[eligible_classes].to_frame("videos"))
print(f"Boxes larger than the {CANVAS_SIDE_PX}px canvas: "
      f"{(objects[['bbox_width', 'bbox_height']].max(axis=1) > CANVAS_SIDE_PX).mean():.2%}")


## 3. Make leakage-resistant train, validation and test splits

The unit of separation is the complete source video. The search below chooses a deterministic assignment that approximately preserves class distributions while heavily penalising missing classes. All frames and objects from one video remain in one split.

In [ ]:
def class_distribution(frame, class_names):
    counts = frame["category_name"].value_counts().reindex(class_names, fill_value=0).astype(float)
    return counts / max(counts.sum(), 1.0)


def find_group_split(frame, seed=42, attempts=5000):
    videos = sorted(frame["video"].unique())
    if len(videos) < 6:
        raise ValueError("At least six source videos are required for a three-way group split.")

    class_names = sorted(frame["category_name"].unique())
    overall = class_distribution(frame, class_names)
    target = {"train": 0.70, "val": 0.15, "test": 0.15}
    n_val = max(1, round(len(videos) * target["val"]))
    n_test = max(1, round(len(videos) * target["test"]))
    rng = np.random.default_rng(seed)
    best = None

    for _ in range(attempts):
        shuffled = list(rng.permutation(videos))
        val_videos = set(shuffled[:n_val])
        test_videos = set(shuffled[n_val:n_val + n_test])
        train_videos = set(shuffled[n_val + n_test:])
        assignment = {"train": train_videos, "val": val_videos, "test": test_videos}

        score = 0.0
        for split_name, split_videos in assignment.items():
            subset = frame[frame["video"].isin(split_videos)]
            present = set(subset["category_name"].unique())
            score += 1000.0 * len(set(class_names) - present)
            score += 10.0 * abs(len(subset) / len(frame) - target[split_name])
            score += float((class_distribution(subset, class_names) - overall).abs().sum())

        if best is None or score < best[0]:
            best = (score, assignment)

    return best


split_score, video_assignment = find_group_split(objects, seed=RANDOM_SEED)
objects["split"] = "unassigned"
for split_name, videos in video_assignment.items():
    objects.loc[objects["video"].isin(videos), "split"] = split_name

assert not (video_assignment["train"] & video_assignment["val"])
assert not (video_assignment["train"] & video_assignment["test"])
assert not (video_assignment["val"] & video_assignment["test"])
assert (objects["split"] != "unassigned").all()

split_path = SPLIT_DIR / "seadronessee_25_35m_object_split.csv"
split_export = objects.drop(columns=["image_exists"]).copy()
split_export["image_path"] = split_export["image_path"].map(
    lambda value: Path(value).resolve().relative_to(REPO_ROOT).as_posix()
)
split_export.to_csv(split_path, index=False)

print(f"Split-search score: {split_score:.3f}")
print(f"Saved: {split_path}")
for split_name in ["train", "val", "test"]:
    videos = sorted(video_assignment[split_name])
    print(f"\n{split_name.upper()} — {len(videos)} videos")
    for video in videos:
        print(" ", video)

display(pd.crosstab(objects["category_name"], objects["split"]))


## 4. Inspect fixed-canvas object patches

Each bounding box is centred in a fixed 640×640 source-pixel canvas. The object is **not resized independently**. This preserves its pixel scale and surrounding context so the later synthetic-altitude transformation can shrink the object relative to the same canvas.

In [ ]:
def fixed_canvas_crop(image, bbox, side=CANVAS_SIDE_PX, fill=CANVAS_FILL):
    x, y, width, height = bbox
    center_x = x + width / 2.0
    center_y = y + height / 2.0
    left = int(round(center_x - side / 2.0))
    top = int(round(center_y - side / 2.0))
    right = left + side
    bottom = top + side

    source_left = max(0, left)
    source_top = max(0, top)
    source_right = min(image.width, right)
    source_bottom = min(image.height, bottom)

    canvas = Image.new("RGB", (side, side), fill)
    if source_right > source_left and source_bottom > source_top:
        region = image.crop((source_left, source_top, source_right, source_bottom))
        canvas.paste(region, (source_left - left, source_top - top))
    return canvas


sample_rows = objects.sample(min(12, len(objects)), random_state=RANDOM_SEED)
fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for axis, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    with Image.open(row["image_path"]) as source:
        patch = fixed_canvas_crop(
            source.convert("RGB"),
            (row["bbox_x"], row["bbox_y"], row["bbox_width"], row["bbox_height"]),
        )
    axis.imshow(patch)
    axis.set_title(f"{row['category_name']}\n{row['altitude_m']:.1f} m | {row['split']}")
    axis.axis("off")
for axis in axes.flat[len(sample_rows):]:
    axis.axis("off")
plt.tight_layout()


## 5. Load the frozen DINOv2 encoder and extract features

Only the linear probe will be trained. DINOv2 remains in evaluation mode and every encoder parameter has `requires_grad=False`. Features are cached so the classifier can be retrained without running the encoder again.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad_(False)

trainable_encoder_parameters = sum(parameter.numel() for parameter in encoder.parameters() if parameter.requires_grad)
print("Model:", MODEL_NAME)
print("Trainable encoder parameters:", trainable_encoder_parameters)
assert trainable_encoder_parameters == 0


In [ ]:
class SeaDronesSeeObjectDataset(Dataset):
    def __init__(self, frame, processor, label_to_index):
        self.frame = frame.reset_index(drop=True).copy()
        self.processor = processor
        self.label_to_index = label_to_index

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(row["image_path"]) as source:
            patch = fixed_canvas_crop(
                source.convert("RGB"),
                (row["bbox_x"], row["bbox_y"], row["bbox_width"], row["bbox_height"]),
            )
        pixel_values = self.processor(images=patch, return_tensors="pt")["pixel_values"].squeeze(0)
        label = self.label_to_index[row["category_name"]]
        return pixel_values, label, index


class_names = sorted(objects["category_name"].unique())
label_to_index = {name: index for index, name in enumerate(class_names)}
index_to_label = {index: name for name, index in label_to_index.items()}
print(label_to_index)


In [ ]:
@torch.inference_mode()
def extract_features(split_name):
    split_frame = objects[objects["split"] == split_name].reset_index(drop=True).copy()
    dataset = SeaDronesSeeObjectDataset(split_frame, processor, label_to_index)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    features, labels, row_indices = [], [], []
    for pixel_values, batch_labels, batch_indices in tqdm(loader, desc=f"Encoding {split_name}"):
        pixel_values = pixel_values.to(DEVICE, non_blocking=True)
        if DEVICE.type == "cuda":
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                output = encoder(pixel_values=pixel_values)
        else:
            output = encoder(pixel_values=pixel_values)

        batch_features = getattr(output, "pooler_output", None)
        if batch_features is None:
            batch_features = output.last_hidden_state[:, 0]
        batch_features = F.normalize(batch_features.float(), dim=1)

        features.append(batch_features.cpu().numpy())
        labels.append(batch_labels.numpy())
        row_indices.append(batch_indices.numpy())

    features = np.concatenate(features)
    labels = np.concatenate(labels)
    row_indices = np.concatenate(row_indices)
    metadata = split_frame.iloc[row_indices].reset_index(drop=True)

    np.savez_compressed(
        FEATURE_DIR / f"{split_name}.npz",
        features=features,
        labels=labels,
    )
    metadata.to_csv(FEATURE_DIR / f"{split_name}_metadata.csv", index=False)
    print(split_name, features.shape)
    return features, labels, metadata


feature_sets = {
    split_name: extract_features(split_name)
    for split_name in ["train", "val", "test"]
}


## 6. Train and select the linear probe

The validation set selects the logistic-regression regularisation strength. After selection, the final probe is fitted on training plus validation features and evaluated once on the untouched test videos.

In [ ]:
X_train, y_train, meta_train = feature_sets["train"]
X_val, y_val, meta_val = feature_sets["val"]
X_test, y_test, meta_test = feature_sets["test"]

candidate_c = [0.01, 0.1, 1.0, 10.0]
validation_results = []
for c_value in candidate_c:
    probe = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=c_value,
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_SEED,
        ),
    )
    probe.fit(X_train, y_train)
    prediction = probe.predict(X_val)
    macro_f1 = f1_score(
        y_val,
        prediction,
        labels=np.arange(len(class_names)),
        average="macro",
        zero_division=0,
    )
    validation_results.append({"C": c_value, "macro_f1": macro_f1})

validation_table = pd.DataFrame(validation_results).sort_values("macro_f1", ascending=False)
display(validation_table)
best_c = float(validation_table.iloc[0]["C"])
print("Selected C:", best_c)

final_probe = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=best_c,
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    ),
)
final_probe.fit(np.concatenate([X_train, X_val]), np.concatenate([y_train, y_val]))
y_pred = final_probe.predict(X_test)


In [ ]:
labels = np.arange(len(class_names))
macro_f1 = f1_score(y_test, y_pred, labels=labels, average="macro", zero_division=0)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
report = classification_report(
    y_test,
    y_pred,
    labels=labels,
    target_names=class_names,
    zero_division=0,
    output_dict=True,
)

print(f"Test macro-F1:          {macro_f1:.4f}")
print(f"Test balanced accuracy: {balanced_accuracy:.4f}")
display(pd.DataFrame(report).T)

matrix = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
fig, axis = plt.subplots(figsize=(8, 7))
sns.heatmap(
    matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axis,
)
axis.set_xlabel("Predicted class")
axis.set_ylabel("True class")
axis.set_title("30 m held-out-video confusion matrix (row normalised)")
plt.tight_layout()


## 7. Cluster-bootstrap uncertainty and save the baseline

Objects from one image are correlated. The interval below resamples complete test images rather than individual object rows. This remains a pilot estimate; the number of independent test flights should be reported alongside it.

In [ ]:
def clustered_macro_f1_interval(y_true, y_prediction, groups, iterations=2000, seed=42):
    y_true = np.asarray(y_true)
    y_prediction = np.asarray(y_prediction)
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    values = []

    group_indices = {group: np.flatnonzero(groups == group) for group in unique_groups}
    for _ in range(iterations):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        values.append(
            f1_score(
                y_true[indices],
                y_prediction[indices],
                labels=np.arange(len(class_names)),
                average="macro",
                zero_division=0,
            )
        )
    return np.quantile(values, [0.025, 0.975])


ci_low, ci_high = clustered_macro_f1_interval(
    y_test,
    y_pred,
    meta_test["image_key"].to_numpy(),
    seed=RANDOM_SEED,
)
print(f"Image-cluster bootstrap 95% CI for macro-F1: [{ci_low:.4f}, {ci_high:.4f}]")

artifact = {
    "probe": final_probe,
    "model_name": MODEL_NAME,
    "class_names": class_names,
    "label_to_index": label_to_index,
    "altitude_range_m": [ALTITUDE_MIN_M, ALTITUDE_MAX_M],
    "canvas_side_px": CANVAS_SIDE_PX,
    "minimum_frame_gap": MIN_FRAME_GAP,
    "selected_C": best_c,
}
joblib.dump(artifact, RESULT_DIR / "linear_probe_30m.joblib")

metrics = {
    "macro_f1": float(macro_f1),
    "balanced_accuracy": float(balanced_accuracy),
    "macro_f1_image_cluster_bootstrap_95ci": [float(ci_low), float(ci_high)],
    "test_objects": int(len(y_test)),
    "test_images": int(meta_test["image_key"].nunique()),
    "test_videos": int(meta_test["video"].nunique()),
    "classification_report": report,
    "validation_search": validation_results,
}
with (RESULT_DIR / "metrics_30m.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print("Saved probe:", RESULT_DIR / "linear_probe_30m.joblib")
print("Saved metrics:", RESULT_DIR / "metrics_30m.json")


## Interpretation and next stage

This notebook establishes only the real 30 m baseline:

```text
real 30 m fixed-canvas crops
        ↓
frozen DINOv2 encoder
        ↓
30 m feature vectors
        ↓
trained linear probe
```

The next notebook should use a disjoint 30 m calibration subset to create paired synthetic 80 m canvases, fit an orthogonal matrix in the frozen feature space, and apply that matrix before this fixed probe:

```text
real 80 m crop → frozen encoder → synthetic-fitted R → fixed 30 m probe
```

For that final test, use complete held-out high-altitude videos and compare against the uncorrected 30 m probe and ordinary scale augmentation. Do not use the real high-altitude test set to select the synthetic transformation.